<a href="https://colab.research.google.com/github/chetools/CHE4071_Spring2026/blob/main/nnx_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -U flax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.7/516.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.5/180.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempting uninstall: jax
    Found existing installation: jax 0.7.2
    Uninstalling jax-0.7.2:
      Successfully uninstalled jax-0.7.2
  Attempting uninstall: flax
    Found existing installation: flax 0.11.2
    Uninstalling flax-0.11.2:
      Successfully uninstalled flax-0.11.2


In [1]:
from flax import nnx
import jax
import jax.numpy as jnp
import optax
from plotly.subplots import make_subplots

In [2]:
class NN(nnx.Module):
    def __init__(self, N, rngs):
        self.norm0 = nnx.BatchNorm(1,rngs=rngs)
        self.l1 = nnx.Linear(1, N, rngs=rngs)
        self.l2 = nnx.Linear(N, N, rngs=rngs)
        self.l3 = nnx.Linear(N, 1, rngs=rngs)

    def __call__(self, x):
        x = self.l1(self.norm0(x))
        x = jax.nn.silu(x)
        x = self.l2(x)
        x = jax.nn.silu(x)
        x = self.l3(x)
        return x

In [3]:
nn=NN(N=10, rngs = nnx.Rngs(0))
nnopt = nnx.Optimizer(nn, optax.adam(1e-3), wrt=nnx.Param)
nngraph, nnstate = nnx.split((nn, nnopt))

In [4]:
x = jnp.linspace(0,10*jnp.pi, 100).reshape(-1,1)
ydata = jnp.sin(x)

In [13]:
@jax.jit
def train(nnstate, x, ydata):
    nn, nnopt = nnx.merge(nngraph, nnstate)
    def loss_fn(nn):
        y_pred = nn(x)
        return jnp.mean((y_pred - ydata) ** 2)

    loss, grad = jax.value_and_grad(loss_fn)(nn)
    nnopt.update(nn, grad)
    nnstate = nnx.state((nn, nnopt))
    return nnstate, loss


In [14]:
for i in range(10000):
    nnstate, val = train(nnstate, x, ydata)
    if i % 1000 == 0:
        print(f"Step {i}, Loss: {val}")

Step 0, Loss: 0.00020778637554030865
Step 1000, Loss: 0.0001912125153467059
Step 2000, Loss: 0.00017685157945379615
Step 3000, Loss: 0.00016424786008428782
Step 4000, Loss: 0.00015451815852429718
Step 5000, Loss: 0.00015366305888164788
Step 6000, Loss: 0.00014626432675868273
Step 7000, Loss: 0.00012864670134149492
Step 8000, Loss: 0.00012242137745488435
Step 9000, Loss: 0.00011697356239892542


In [15]:
nn, nnopt = nnx.merge(nngraph, nnstate)

In [16]:
fig = make_subplots()
fig.add_scatter(x=jnp.squeeze(x), y=jnp.squeeze(ydata), mode='markers', name='Data')
fig.add_scatter(x=jnp.squeeze(x), y=jnp.squeeze(nn(x)), mode='lines')
fig.update_layout(template='plotly_dark')

In [ ]:
f